# Batch Runner — Google Colab

## Etapes AVANT d'ouvrir ce notebook

**1. Uploader le projet sur Google Drive**
- Sur ton PC : compresse `ProjetLangGraph_Willy/` en zip
- Upload le zip dans ton Google Drive à la racine (`Mon Drive/ProjetLangGraph_Willy.zip`)

**2. Ajouter les secrets dans Colab** (icone clé dans le panneau gauche) :
```
KONG_CHAT_URL       → valeur dans ton .env
KONG_EMBED_URL      → valeur dans ton .env
KONG_API_KEY        → valeur dans ton .env
PINECONE_API_KEY    → valeur dans ton .env
PINECONE_HOST       → valeur dans ton .env
SHEET_ID            → valeur dans ton .env
SHEET_WORKSHEET     → ex: Langgraph_G3.5_2606
```

**3. Executer les cellules dans l'ordre**

## Cellule 1 — Monter Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print('Drive monte')

## Cellule 2 — Dezipper le projet

In [ ]:
import os

# Modifie ce chemin si ton zip est ailleurs dans Drive
DRIVE_ZIP  = '/content/drive/MyDrive/ProjetLangGraph_Willy.zip'
LOCAL_ROOT = '/content/ProjetLangGraph_Willy'

if not os.path.exists(DRIVE_ZIP):
    raise FileNotFoundError(f'Zip introuvable : {DRIVE_ZIP}\nVerifie le chemin dans Drive.')

if not os.path.exists(LOCAL_ROOT):
    print('Dezippage...')
    os.system(f'unzip -q "{DRIVE_ZIP}" -d /content/')
else:
    print('Dossier deja present, dezippage ignore')

os.chdir(LOCAL_ROOT)
print(f'Repertoire courant : {os.getcwd()}')
os.system('ls')

## Cellule 3 — Installer les dependances

In [ ]:
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-e', '.', '--quiet'])
print('Package agentic4api installe')

## Cellule 4 — Charger les secrets et configurer l'environnement

In [ ]:
import os
from google.colab import userdata

SECRETS = [
    'KONG_CHAT_URL', 'KONG_EMBED_URL', 'KONG_API_KEY',
    'PINECONE_API_KEY', 'PINECONE_HOST',
    'SHEET_ID', 'SHEET_WORKSHEET',
]

missing = []
for key in SECRETS:
    try:
        os.environ[key] = userdata.get(key)
        print(f'  OK       {key}')
    except Exception:
        missing.append(key)
        print(f'  MANQUANT {key}')

if missing:
    raise EnvironmentError(f'Ajoute ces secrets dans le panneau cle Colab : {missing}')

# Valeurs par defaut (pas de secret necessaire)
os.environ.setdefault('GOOGLE_AUTH_MODE',  'oauth2')
os.environ.setdefault('GEMINI_MODEL',      'gemini-3.5-flash')
os.environ.setdefault('TOOL_MODE',         'text')
os.environ.setdefault('DEBUG_MODE',        'false')
os.environ.setdefault('BATCH_WAIT_S',      '1')
os.environ.setdefault('MAX_OUTPUT_TOKENS', '1024')
os.environ.setdefault('TOP_K',             '10')
os.environ.setdefault('KONG_VERIFY_SSL',   'false')

print('\nEnvironnement configure')

## Cellule 5 — Authentification Google Sheets (Colab)

In [ ]:
from google.colab import auth
auth.authenticate_user()

import gspread
from google.auth import default

creds, _ = default()

# Remplace _client() de sheet_writer par l'auth Colab
# (evite d'avoir besoin de token.json / credentials.json)
import agentic4api.batch.sheet_writer as _sw
_sw._client = lambda: gspread.authorize(creds)

print('Auth Google Sheets OK')

## Cellule 6 — Preparer les logs et verifier le golden dataset

In [ ]:
import os
os.makedirs('logs', exist_ok=True)

from agentic4api.batch.golden import load_golden
golden = load_golden()
print(f'Golden dataset : {len(golden)} questions')
print(f'Worksheet cible : {os.environ["SHEET_WORKSHEET"]}')

## Cellule 7 — Lancer le batch

Mets `LIMIT = 5` pour un smoke test avant de lancer les 464 questions.

In [ ]:
import os
from agentic4api.batch.run_batch import run
from agentic4api.graph.build import build_graph

# ── Config du run ─────────────────────────────────────────────
LIMIT  = 5     # None = toutes les 464 questions
SKIP   = 0     # reprendre a partir de la question N
# ─────────────────────────────────────────────────────────────

graph = build_graph(use_memory=False)

run(
    worksheet   = os.environ['SHEET_WORKSHEET'],
    limit       = LIMIT,
    skip        = SKIP,
    parallel    = False,
    batch_size  = 1,
    resume      = False,
    resume_from = None,
    graph       = graph,
)

## Cellule 8 — (Optionnel) Sauvegarder les logs sur Drive

In [ ]:
import shutil, datetime

DEST = f'/content/drive/MyDrive/batch_logs_{datetime.date.today()}'
shutil.copytree('logs', DEST, dirs_exist_ok=True)
print(f'Logs sauvegardes dans Drive : {DEST}')